In [1]:
import numpy as np
from abc import ABC, abstractmethod

In [2]:
class Layer(ABC):
    @abstractmethod
    def __init__(self):
        pass
    
    @abstractmethod
    def forward(self, x):
        pass
    
    @abstractmethod
    def backward(self, upstream_grad):
        pass

In [3]:
class MathematicalOperator:
    @abstractmethod
    def __init__(self):
        pass

    def forward(self, x):
        pass

    def backward(self, upstream_grad):
        pass

In [4]:
class GradTensor:
    def __init__(self, tensor):
        self.data = tensor
        self.shape = self.data.shape
        self.grad = None
    
    def _zero_grad(self):
        self.grad = None

In [5]:
class LinearLayer(Layer):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = GradTensor(np.random.rand(
            self.in_features,
            self.out_features
        ))

        self.bias = GradTensor(np.zeros(
            self.out_features
        ))
    
    def forward(self, x):
        self.x = x
        return np.matmul(self.x, self.weight.data) + self.bias.data
    
    def backward(self, upstream_grad):
        self.weight.grad = np.matmul(self.x.T, upstream_grad) # (I x O) --> (I x B) @ (B x O)
        self.bias.grad = np.sum(upstream_grad, axis = 0, keepdims=True)
        return np.matmul(upstream_grad, self.weight.grad.T)
    
    def __call__(self, x):
        return self.forward(x)

In [ ]:
class NaiveSoftmax(MathematicalOperator):
    def __init__(self):
        pass

    def forward(self, x):
        # batch, d_head, seq_length, n_classes = x.shape
        adj_exp_x = np.exp(x - np.max(x, axis = 1), keepdims = True)
        self.probs = adj_exp_x / np.sum(adj_exp_x, keepdims=True, axis = 1)
        return self.probs
    
    def __call__(self, x):
        return self.forward(x)
    
    def backward(self, upstream_grad):
        """
        Computes del(Softmax)/del(input_to_softmax): each input to the softmax will tell us how much they effected the Softmax
        Therefore we can say that, shape of Derivative of Softmax w.r.t its input is self.x.shape
        """
        self.grad_tensor = np.zeros_like(self.upstream_grad)
        for i in range(len(self.grad_tensor)):
            self.grad_tensor[i] = - self.probs[i].reshape(1, -1) * self.probs[i].reshape(-1, 1)
        
        for i in range(len(self.grad_tensor)):
            for j in range(len(self.grad_tensor)):
                if i == j:
                    self.grad_tensor[i, j] = self.probs[i] * (1 - self.peobs[i])
        
        return np.matmul(self.grad_tensor, upstream_grad)

In [7]:
class LayerNorm(Layer):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return super().forward(x)
    
    def backward(self, grad):
        return super().backward(grad) 

In [8]:
batch = 10
in_features = 3
out_features = 2
x = np.random.rand(batch, in_features)
layer_1 = LinearLayer(in_features, out_features)
logits = layer_1.forward(x)
layer_1.backward(np.random.rand(batch, out_features))

array([[2.65522812, 2.70686678, 2.24647302],
       [3.93386176, 4.08802731, 3.24946795],
       [5.60146928, 5.86783604, 4.57941614],
       [1.81580309, 1.88830481, 1.49853739],
       [4.15091973, 4.32142183, 3.42081798],
       [3.97546924, 4.15625998, 3.25847483],
       [3.36955205, 3.53867673, 2.74571515],
       [4.66081162, 4.84808574, 3.8452589 ],
       [4.25335957, 4.42218952, 3.51120613],
       [1.29592333, 1.33707596, 1.08024037]])

In [9]:
NaiveSoftmax()(logits)

array([[0.51443304, 0.48556696],
       [0.49579762, 0.50420238],
       [0.4951067 , 0.5048933 ],
       [0.51680878, 0.48319122],
       [0.49469971, 0.50530029],
       [0.47866016, 0.52133984],
       [0.50531972, 0.49468028],
       [0.47445666, 0.52554334],
       [0.51822405, 0.48177595],
       [0.51226988, 0.48773012]])

In [16]:
from sympy import Matrix, symbols

# Define variables for X (batch=2, features=3)
x1, x2, x3, x4, x5, x6, x7, x8, x9 = symbols("x1 x2 x3 x4 x5 x6 x7 x8 x9")

X = Matrix([
    [x1, x2, x3],
    [x4, x5, x6],
    [x7, x8, x9]
])

# Define variables for W (3 × 2)
w1, w4, w2, w5, w3, w6 = symbols("w1 w4 w2 w5 w3 w6")

W = Matrix([
    [w1, w4],
    [w2, w5],
    [w3, w6]
])

# Define bias (2,)
b1, b2 = symbols("b1 b2")
b = Matrix([[b1, b2]])

In [18]:
Y = X * W + Matrix([
    [b1, b2],
    [b1, b2],
    [b1, b2]
])
Y

Matrix([
[b1 + w1*x1 + w2*x2 + w3*x3, b2 + w4*x1 + w5*x2 + w6*x3],
[b1 + w1*x4 + w2*x5 + w3*x6, b2 + w4*x4 + w5*x5 + w6*x6],
[b1 + w1*x7 + w2*x8 + w3*x9, b2 + w4*x7 + w5*x8 + w6*x9]])